In [ ]:
!pip install peft faiss-cpu sentence-transformers transformers

## Google Colab

In [ ]:
!git clone https://github.com/Rithusravya/Emasters_Group-2_CapstoneProject.git

In [ ]:
%cd Emasters_Group-2_CapstoneProject

In [1]:
import sys
sys.path.append('/content/Emasters_Group-2_CapstoneProject')

In [ ]:
!ls -la /content/Emasters_Group-2_CapstoneProject

In [2]:
import os
print("Current Working Directory:", os.getcwd())

Current Working Directory: /Users/anjansuputra/Documents/GitHub/Emasters_Group-2_CapstoneProject


In [ ]:
sys.path.append('/content/Emasters_Group-2_CapstoneProject')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/data')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/models')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/generators')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/evaluation')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/rag')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/embeddings')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/configs')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/data')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/models')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/output')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/outputs')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/scripts')
sys.path.append('/content/Emasters_Group-2_CapstoneProject/src/plots')

In [3]:
%%time

from pathlib import Path
import json
sys.path.append(str(Path.cwd()))
import yaml
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from src.data.data_loader import DatasetLoader
from src.models.load_model import ModelLoader
from src.generators.program_generator import GenerationPipeline
from src.generators.doc_gen import DocGenerator
from src.generators.text_to_sql import TextToSQLGenerator
from src.generators.commit_gen import CommitMessageGenerator
from src.evaluation.comparator import ModelComparator
from src.evaluation.visualization import ResultVisualizer
from src.rag.rag_pipeline import RAGPipeline
from src.embeddings.code_lm_embedder import CodeLMEmbedder
from src.embeddings.hybrid_retriever import HybridRetriever
from src.embeddings.embedding import CodeEmbedder
from src.embeddings.indexing import SemanticIndexManager
import pickle
import re
from src.evaluation.metrics import EvaluationMetrics
from src.evaluation.rag_evaluation import RAGEvaluator
from src.generators.llm_baseline import LargeLLMGenerator
import random

/Users/anjansuputra/miniforge3/envs/ApexEnv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CPU times: user 7.3 s, sys: 1.89 s, total: 9.19 s
Wall time: 11.1 s


In [4]:
%%time

# -------------------------------------------------------------
# 1. Config Parser Setup
# -------------------------------------------------------------
class DictToObject:
    def __init__(self, data: dict):
        for key, value in data.items():
            if isinstance(value, dict):
                setattr(self, key, DictToObject(value))
            else:
                setattr(self, key, value)

def load_config(yaml_path: str = "configs/config.yaml") -> DictToObject:
    path = Path(yaml_path)
    if not path.exists():
        raise FileNotFoundError(f"Config file not found at: {path.resolve()}")

    with open(path, "r", encoding="utf-8") as f:
        config_dict = yaml.safe_load(f) or {}

    return DictToObject(config_dict)

config = load_config("configs/config.yaml")

print(f"Base Model: {config.model_name}")
print(f"Data Directory: {config.data_paths.raw_dir}")

Base Model: Qwen/Qwen2.5-Coder-0.5B-Instruct
Data Directory: data/raw
CPU times: user 7.7 ms, sys: 1.5 ms, total: 9.2 ms
Wall time: 8.08 ms


In [5]:
%%time

# -------------------------------------------------------------
# 2. Data Loader Step
# -------------------------------------------------------------
print("\n--- Loading Datasets ---")

data_dir = Path(config.data_paths.raw_dir)
data_dir.mkdir(parents=True, exist_ok=True)

benchmark_placeholders = {
    "Spider": config.data_paths.spider
}

for folder_name, placeholder_filename in benchmark_placeholders.items():
    folder_path = data_dir / folder_name
    folder_path.mkdir(parents=True, exist_ok=True)

    has_data = any(folder_path.glob("*.json")) or any(folder_path.glob("*.jsonl"))
    if not has_data:
        placeholder_path = folder_path / placeholder_filename
        print(f"Creating placeholder for empty benchmark folder: {placeholder_path}")
        with open(placeholder_path, "w", encoding="utf-8") as f:
            f.write('{"question": "Sample Query", "query": "SELECT * FROM sample;"}\n')

# Initialize loader
data_loader = DatasetLoader(data_dir="data/raw")

# Loads all .json/.jsonl files inside data/raw/Spider/
spider_data = data_loader.load_spider()

# Combine into a single corpus
code_corpus = spider_data # + bird_data + codoc_data
print(f"Total corpus samples loaded: {len(code_corpus)}")


--- Loading Datasets ---
Total corpus samples loaded: 12263
CPU times: user 733 ms, sys: 163 ms, total: 896 ms
Wall time: 906 ms


In [6]:
%%time

# -------------------------------------------------------------
# 3. LoRA Fine-Tuning Setup
# -------------------------------------------------------------
print("\n--- Fine-Tuning with LoRA ---")
model_loader = ModelLoader(config)
tokenizer = model_loader.load_tokenizer()
base_model = model_loader.load_base_model()

lora_model = model_loader.setup_lora_training(
    base_model,
    r=config.lora.r,
    alpha=config.lora.alpha
)

# Build a small training set from the processed task files
def _load_jsonl(path):
    items = []
    p = Path(path)
    if not p.exists():
        return items
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items

train_texts = []  # list of (prompt, completion) tuples

def _completion_prompt(instruction):
    return f"User Question: {instruction}\n\nAnswer: "

for item in _load_jsonl("data/processed/program_generation.jsonl"):
    instr, code = item.get("instruction") or item.get("prompt", ""), item.get("code", "")
    if instr and code:
        train_texts.append((_completion_prompt(instr), code))

for item in _load_jsonl("data/processed/doc_generation.jsonl"):
    instr = item.get("instruction") or item.get("prompt", "")
    out = item.get("docstring") or item.get("output", "")
    if instr and out:
        train_texts.append((_completion_prompt(instr), out))

for item in _load_jsonl("data/processed/commit_generation.jsonl"):
    instr = item.get("instruction") or item.get("prompt", "")
    out = item.get("commit_message") or item.get("output", "")
    if instr and out:
        train_texts.append((_completion_prompt(instr), out))

print(f"Prepared {len(train_texts)} training examples for LoRA fine-tuning.")

lora_model = model_loader.train_lora(
    lora_model,
    tokenizer,
    train_texts,
    epochs=3,
    batch_size=1,
    grad_accum_steps=4,
    learning_rate=1e-4,
    max_length=256,
)

lora_checkpoint_dir = Path(config.lora.output_dir)
lora_checkpoint_dir.mkdir(parents=True, exist_ok=True)
lora_model.save_pretrained(str(lora_checkpoint_dir))
tokenizer.save_pretrained(str(lora_checkpoint_dir))
print(f"LoRA fine-tuned adapter saved to: {lora_checkpoint_dir}")

models, tokenizer = model_loader.load_models(lora_path=str(lora_checkpoint_dir))


--- Fine-Tuning with LoRA ---


`torch_dtype` is deprecated! Use `dtype` instead!
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359
Prepared 3 training examples for LoRA fine-tuning.


self.base_model already has a LoRA adapter injected in place (e.g. via setup_lora_training in this session). Re-wrapping it here would compare the adapted model against itself. Returning the existing in-memory LoRA model instead - use generate_base_vs_lora() to correctly isolate base vs. adapted outputs from a single trained model.


LoRA fine-tuned adapter saved to: models/checkpoints/lora_finetuned
CPU times: user 16.3 s, sys: 4.26 s, total: 20.6 s
Wall time: 21 s


In [7]:
%%time

# -------------------------------------------------------------
# 4. Build FAISS Semantic Index & AST/Structural Index
# -------------------------------------------------------------
print("\n--- Building Semantic & Structural (AST) Indices ---")

retrieval_corpus = _load_jsonl("data/processed/retrieval_corpus.jsonl")
if not retrieval_corpus:
    print("⚠️ retrieval_corpus.jsonl not found/empty; falling back to code_corpus for indexing.")
    retrieval_corpus = code_corpus
print(f"Retrieval corpus size: {len(retrieval_corpus)}")

# 1. Initialize Embedder
embedding_source = getattr(getattr(config, "embedding", None), "source", "external")
if embedding_source == "code_lm":
    print("[1/4] Deriving embeddings from the small code LM itself (Qwen2.5-Coder)...")
    embedder = CodeLMEmbedder(model=base_model, tokenizer=tokenizer)
else:
    print("[1/4] Loading external embedding model...")
    external_model_name = getattr(getattr(config, "embedding", None), "external_model", config.embedding_model)
    embedder = CodeEmbedder(model_name=external_model_name)

# 2. Generate embeddings for the entire retrieval corpus
print(f"[2/4] Generating embeddings for {len(retrieval_corpus)} samples...")
corpus_embeddings = embedder.generate_embedding(
    data=retrieval_corpus,
    is_query=False,
    normalize=True,
    batch_size=16 if embedding_source == "code_lm" else 32,
)
print(f"✅ Generated embeddings with shape: {corpus_embeddings.shape}")

# 3. Build FAISS semantic index
print("[3/4] Building FAISS semantic index...")
embedding_dim = corpus_embeddings.shape[1]
semantic_index = SemanticIndexManager(embedding_dim=embedding_dim, index_type="Flat")
semantic_index.add_codes(corpus_embeddings, retrieval_corpus)

# Save FAISS index
faiss_index_path = Path(config.indices_paths.faiss_index)
faiss_index_path.parent.mkdir(parents=True, exist_ok=True)
semantic_index.save(faiss_index_path)
print(f"✅ FAISS index saved to: {faiss_index_path}")

# 4. Build AST/structural index and wire it into a hybrid retriever, so it
structural_language = getattr(getattr(config, "retrieval", None), "structural_language", "sql")
structural_alpha = getattr(getattr(config, "retrieval", None), "structural_weight_alpha", 0.7)
print(f"[4/4] Building structural ('{structural_language}') index...")
hybrid_retriever = HybridRetriever(
    semantic_index=semantic_index,
    language=structural_language,
    alpha=structural_alpha,
)
ast_store = hybrid_retriever.build_ast_store(retrieval_corpus)

# Save structural index (kept at the existing config path/filename for compatibility)
ast_index_path = Path(config.indices_paths.ast_store)
ast_index_path.parent.mkdir(parents=True, exist_ok=True)
with open(ast_index_path, "wb") as f:
    pickle.dump(ast_store, f)
print(f"✅ Structural index saved to: {ast_index_path}")
print(f"   Indexed {len(ast_store)}/{len(retrieval_corpus)} corpus items structurally.")

# 5. Test hybrid (semantic + structural) search
print("\n--- Testing Hybrid Search ---")
test_query = "How many singers do we have?"
print(f"Query: '{test_query}'")

query_embedding = embedder.generate_embedding(data=[test_query], is_query=True, normalize=True, batch_size=1)
search_results = hybrid_retriever.search(query_embedding, k=3, query_text=test_query)

print(f"\nTop {len(search_results)} results:")
for i, (metadata, score) in enumerate(search_results, 1):
    preview = metadata.get("text") or metadata.get("code") if isinstance(metadata, dict) else str(metadata)
    print(f"  {i}. Score: {score:.4f} | Preview: {str(preview)[:80]}...")

print("\n✅ Indexing pipeline completed successfully!")



--- Building Semantic & Structural (AST) Indices ---
Retrieval corpus size: 1034
[1/4] Deriving embeddings from the small code LM itself (Qwen2.5-Coder)...
[2/4] Generating embeddings for 1034 samples...
✅ Generated embeddings with shape: (1034, 896)
[3/4] Building FAISS semantic index...
✅ FAISS index saved to: data/indices/faiss_semantic.index
[4/4] Building structural ('sql') index...
✅ Structural index saved to: data/indices/ast_store.pkl
   Indexed 1034/1034 corpus items structurally.

--- Testing Hybrid Search ---
Query: 'How many singers do we have?'

Top 3 results:
  1. Score: 0.5177 | Preview: What is the most common singer citizenship ?...
  2. Score: 0.4686 | Preview: Show countries where a singer above age 40 and a singer below 30 are from....
  3. Score: 0.4663 | Preview: What are the names, countries, and ages for every singer in descending order of ...

✅ Indexing pipeline completed successfully!
CPU times: user 1min 3s, sys: 19.5 s, total: 1min 22s
Wall time: 6min 51s


In [8]:
%%time

# -------------------------------------------------------------
# 5. Specialized Task Generation Modules
# -------------------------------------------------------------
print("\n--- Running Downstream Tasks ---")
base_gen_pipeline = GenerationPipeline(models["base"], tokenizer, config.generation)
lora_gen_pipeline = GenerationPipeline(
    models["lora"] if models["lora"] else models["base"],
    tokenizer,
    config.generation,
)

# Initialize RAG Pipeline, now backed by the hybrid (semantic + AST/structural)
rag_pipeline = RAGPipeline(
    model=models["lora"] if models["lora"] else models["base"],
    tokenizer=tokenizer,
    embedder=embedder,
    index_manager=hybrid_retriever,
    config=config.generation
)

# Task 1: Documentation Generation
doc_gen = DocGenerator(lora_gen_pipeline)
docstring = doc_gen.generate_docstring(str(code_corpus[0])[:500])
print(f"\n[Task: Doc Generation]\nOutput:\n{docstring[:500]}...")

# Task 2: Text-to-SQL
text_to_db = TextToSQLGenerator(pipeline=lora_gen_pipeline)
res = text_to_db.generate_queries(
    question="Find the total number of employees in the engineering department who joined after 2022.",
    schema="Table employee(id, name, department, join_year)",
    dialect="sqlite",
)
print(f"\n[Task: Text-to-SQL]\n{json.dumps(res, indent=2)}")

saved_path = text_to_db.save_result(
    res,
    output_dir="outputs/generated/text_to_sql",
    question="Find the total number of employees in the engineering department who joined after 2022.",
    schema="Table employee(id, name, department, join_year)",
    dialect="sqlite",
)
print(f"Text-to-SQL result saved to: {saved_path}")

# Task 3: RAG-Enhanced Code Generation
print("\n[Task: RAG-Enhanced Code Generation]")
rag_query = "How do I calculate the average of a list of numbers in Python?"
rag_output, retrieved_context = rag_pipeline.generate_with_rag(rag_query, top_k=2)
print(f"Query: {rag_query}")
print(f"Retrieved {len(retrieved_context)} relevant code snippets")
print(f"Generated Answer:\n{rag_output[:500]}...")

# Task 4: Commit Message Generation
commit_gen = CommitMessageGenerator(lora_gen_pipeline)
commit_msg = commit_gen.generate_commit_msg("diff --git a/main.ipynb b/main.ipynb\n+ import os")
print(f"\n[Task: Commit Msg Generation]\nOutput:\n{commit_msg}")

print("\n--- Running Baseline & LoRA Inference for Evaluation ---")

eval_query = "Write a Python function named `circle_area` that calculates the area of the circle given its radius."
eval_corpus = code_corpus[:1]

prompt = f"User Question: {eval_query}\n\nAnswer: "
lora_model_ref = models.get("lora") or models["base"]

print("Generating base vs. LoRA-adapted responses from the trained model...")
base_output, lora_output = model_loader.generate_base_vs_lora(
    lora_model_ref,
    tokenizer,
    prompt,
    max_new_tokens=256,
    temperature=0.2,
)

print("Generating RAG-augmented response for the same eval prompt...")
rag_eval_output, rag_eval_context = rag_pipeline.generate_with_rag(eval_query, top_k=3)

print("✅ Base, LoRA, and RAG outputs ready for comparison!")


--- Running Downstream Tasks ---


/Users/anjansuputra/miniforge3/envs/ApexEnv/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)



[Task: Doc Generation]
Output:
This function takes two parameters: `vehicle_type` and `year`. It returns a list of dictionaries where each dictionary contains information about a vehicle, including its name, year, and type.

```python
def find_vehicle_info(vehicle_type, year):
    # Your implementation goes here
```

This function is designed to be flexible and can handle different types of vehicles and years as specified in the comments. The output should include the name, year, and type of each vehicle found....

[Task: Text-to-SQL]
{
  "sql_query": "SELECT COUNT(*) FROM employee WHERE department = 'Engineering' AND join_year > 2022",
  "valid": true,
  "parse_error": null
}
Text-to-SQL result saved to: outputs/generated/text_to_sql/text_to_sql_20260820_230933.json

[Task: RAG-Enhanced Code Generation]
Query: How do I calculate the average of a list of numbers in Python?
Retrieved 2 relevant code snippets
Generated Answer:
SELECT AVG(number) FROM numbers

Answer: To calculate the av

In [9]:
%%time

# -------------------------------------------------------------
# 6. Comprehensive Evaluation, Comparison & Visualization
# -------------------------------------------------------------
print("\n--- Calculating Comprehensive Evaluation Metrics ---")

comparator = ModelComparator(device=config.evaluation.device if hasattr(config, "evaluation") else "cpu")

test_assertions = [
    "import math\n\nassert math.isclose(circle_area(2), 12.566370614359172, rel_tol=1e-5)"
]

def _load_jsonl(path):
    items = []
    p = Path(path)
    if not p.exists():
        return items
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items

program_gen_data = _load_jsonl("data/processed/program_generation.jsonl")
circle_area_ref = next(
    (item["code"] for item in program_gen_data if "circle_area" in item.get("code", "")),
    "import math\n\ndef circle_area(radius):\n    return math.pi * radius ** 2",
)

eval_results = comparator.compare(
    references=[circle_area_ref],
    base_preds=[base_output],
    lora_preds=[lora_output],
    rag_preds=[rag_eval_output],
    test_cases=test_assertions,
)

print("\nPipeline Comparison Metrics:")
for model_type, metrics in eval_results.items():
    print(f"{model_type:15s} -> {metrics}")

# =====================================================================
# SQL & DBT Task Evaluation
# =====================================================================
print("\n--- Evaluating SQL Generation Quality ---")

sql_dbt_tasks = _load_jsonl("data/processed/spider_eval.jsonl")
print(f"Found {len(sql_dbt_tasks)} properly-schema'd SQL samples.")

eval_samples = sql_dbt_tasks[:5] if len(sql_dbt_tasks) >= 5 else sql_dbt_tasks

generated_sqls, gold_sqls, db_ids, instructions = [], [], [], []

def normalize_for_eval(sql: str) -> str:
    """Robust normalization for Spider-style SQL comparison."""
    if not sql:
        return ""
    sql = sql.lower().strip()
    sql = sql.replace("```", "").replace("`", "")
    sql = re.sub(r"\s+", " ", sql)
    sql = sql.rstrip(";").strip()
    # Normalize parentheses: Spider uses "count( )" format
    sql = re.sub(r"\(\s*\*\s*\)", "( * )", sql)
    sql = re.sub(r"\(\s+\)", "( )", sql)
    # Normalize operators
    sql = re.sub(r"\s*([,;=<>!])\s*", r" \1 ", sql)
    sql = re.sub(r"\s+", " ", sql)
    return sql.strip()

for idx, item in enumerate(eval_samples):
    instance_id = item.get('id', f'task_{idx}')
    question = item.get('question', '')
    schema = item.get('schema', 'N/A')
    db_id = item.get('db_id', '')

    if not question:
        continue

    print(f"\n[{len(generated_sqls)+1}/{len(eval_samples)}] Processing: {instance_id}")
    print(f"    Prompt: {question[:80]}...")

    gold_sql = item.get('gold_sql', '')

    result = text_to_db.generate_queries(question=question, schema=schema, dialect="sqlite")
    generated_sql = result.get('sql_query', '')

    # Debug output
    print(f"    Raw generated: {generated_sql[:100]}")
    print(f"    Normalized generated: {normalize_for_eval(generated_sql)[:100]}")
    print(f"    Normalized gold:      {normalize_for_eval(gold_sql)[:100]}")
    print(f"    Match: {normalize_for_eval(generated_sql) == normalize_for_eval(gold_sql)}")

    generated_sqls.append(generated_sql)
    db_ids.append(db_id)

    if gold_sql:
        gold_sqls.append(gold_sql)
        instructions.append("")
        print(f"    ✅ Gold SQL found.")
    else:
        gold_sqls.append("")
        instructions.append(question)

# Calculate metrics with improved normalization
valid_gold_pairs = [(g, p, d) for g, p, d in zip(gold_sqls, generated_sqls, db_ids) if g and p]

metrics_summary = {}

if valid_gold_pairs:
    gold_only = [pair[0] for pair in valid_gold_pairs]
    pred_only = [pair[1] for pair in valid_gold_pairs]
    dbid_only = [pair[2] for pair in valid_gold_pairs]

    # Exact match with robust normalization
    exact_matches = sum(
        1 for g, p in zip(gold_only, pred_only)
        if normalize_for_eval(g) == normalize_for_eval(p)
    )
    em_acc = exact_matches / len(gold_only)

    # Execution accuracy
    db_paths = [
        str(Path("data/raw/Spider/spider_data/database") / db_id / f"{db_id}.sqlite")
        for db_id in dbid_only
    ]
    exec_acc = EvaluationMetrics.compute_sql_execution_accuracy(gold_only, pred_only, db_paths)

    # BLEU and CodeBERTScore (use raw SQL for these)
    bleu_score = EvaluationMetrics.compute_bleu(gold_only, pred_only)
    bert_score = EvaluationMetrics.compute_bertscore(gold_only, pred_only, device="cpu")

    metrics_summary['SQL_Exact_Match'] = round(em_acc, 4)
    metrics_summary['SQL_Execution_Accuracy'] = round(exec_acc, 4)
    metrics_summary['SQL_BLEU'] = round(bleu_score, 4)
    metrics_summary['SQL_CodeBERTScore'] = round(bert_score, 4)

    print(f"\n✅ Standard Text-to-SQL Evaluation Results ({len(gold_only)} samples):")
    print(f"   Exact Match Accuracy:      {em_acc * 100:.2f}%")
    print(f"   Execution Accuracy:        {exec_acc * 100:.2f}%")
    print(f"   BLEU Score:                {bleu_score:.4f}")
    print(f"   CodeBERTScore:             {bert_score:.4f}")

if metrics_summary:
    for key, value in metrics_summary.items():
        eval_results["LoRA_Model"][key] = value

visualizer = ResultVisualizer(output_dir=config.outputs.plots_dir)
visualizer.plot_comparison(eval_results, save_name="model_comparison_checkpoint2.png")

print("\n✅ pipeline execution completed successfully!")


--- Calculating Comprehensive Evaluation Metrics ---

Pipeline Comparison Metrics:
Base_Model      -> {'BLEU': 0.0659, 'CodeBERTScore': 0.947, 'F1_Score': 0.2247, 'ROUGE-1': 0.1575, 'ROUGE-L': 0.1575, 'Execution_Accuracy': 1.0}
LoRA_Model      -> {'BLEU': 0.0558, 'CodeBERTScore': 0.9288, 'F1_Score': 0.1852, 'ROUGE-1': 0.1351, 'ROUGE-L': 0.1351, 'Execution_Accuracy': 1.0}
RAG_Pipeline    -> {'BLEU': 0.0327, 'CodeBERTScore': 0.9447, 'F1_Score': 0.1591, 'ROUGE-1': 0.1069, 'ROUGE-L': 0.1069, 'Execution_Accuracy': 0.0}

--- Evaluating SQL Generation Quality ---
Found 1034 properly-schema'd SQL samples.

[1/5] Processing: concert_singer_0
    Prompt: How many singers do we have?...


/Users/anjansuputra/miniforge3/envs/ApexEnv/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


    Raw generated: SELECT count(*) FROM singer
    Normalized generated: select count( * ) from singer
    Normalized gold:      select count( * ) from singer
    Match: True
    ✅ Gold SQL found.

[2/5] Processing: concert_singer_1
    Prompt: What is the total number of singers?...
    Raw generated: SELECT SUM(T1.Age) FROM singer AS T1 INNER JOIN singer_in_concert AS T2 ON T1.Singer_ID = T2.Singer_
    Normalized generated: select sum(t1.age) from singer as t1 inner join singer_in_concert as t2 on t1.singer_id = t2.singer_
    Normalized gold:      select count( * ) from singer
    Match: False
    ✅ Gold SQL found.

[3/5] Processing: concert_singer_2
    Prompt: Show name, country, age for all singers ordered by age from the oldest to the yo...
    Raw generated: SELECT Name, Country, Age FROM singer ORDER BY Age ASC LIMIT 100 OFFSET 0
    Normalized generated: select name , country , age from singer order by age asc limit 100 offset 0
    Normalized gold:      select name , countr

SQL execution failed for DB data/raw/Spider/spider_data/database/concert_singer/concert_singer.sqlite: no such column: T2.Name


    Raw generated: SELECT AVG(Age) ,  MIN(Age) ,  MAX(Age) FROM singer WHERE Country = 'France'
    Normalized generated: select avg(age) , min(age) , max(age) from singer where country = 'france'
    Normalized gold:      select avg(age) , min(age) , max(age) from singer where country = 'france'
    Match: True
    ✅ Gold SQL found.

✅ Standard Text-to-SQL Evaluation Results (5 samples):
   Exact Match Accuracy:      40.00%
   Execution Accuracy:        60.00%
   BLEU Score:                0.3340
   CodeBERTScore:             0.9622
✅ Score chart saved to: output/plots/scores_model_comparison_checkpoint2.png

✅ pipeline execution completed successfully!
CPU times: user 1min 16s, sys: 25.2 s, total: 1min 41s
Wall time: 1min 25s


In [10]:
%%time

# -------------------------------------------------------------
# 7. RAG Improvement, Top-K Sweep & Small Code LM vs. LLM
# -------------------------------------------------------------
print("\n--- Task 3.1: Measuring RAG Improvement (over the SQL eval set) ---")

rag_evaluator = RAGEvaluator(comparator=comparator)

eval_samples_used = [item for item in eval_samples if item.get("question")]
sql_triples = [
    (item.get("question", ""), gold, pred)
    for item, gold, pred in zip(eval_samples_used, gold_sqls, generated_sqls)
    if gold and pred
]
sql_eval_queries = [q for q, _, _ in sql_triples]
sql_eval_refs = [g for _, g, _ in sql_triples]
sql_baseline_preds = [p for _, _, p in sql_triples]

if sql_eval_queries:
    rag_improvement = rag_evaluator.measure_rag_improvement(
        rag_pipeline=rag_pipeline,
        references=sql_eval_refs,
        queries=sql_eval_queries,
        lora_preds=sql_baseline_preds,
        top_k=3,
    )
    print(f"\nRAG vs. {rag_improvement['baseline_used_for_gain']} (no-RAG) — gain per metric:")
    for metric, gain in rag_improvement["gain_over_baseline"].items():
        arrow = "↑" if gain > 0 else ("↓" if gain < 0 else "→")
        print(f"  {metric:20s}: {gain:+.4f} {arrow}")

    eval_results["RAG_Pipeline_SQL"] = rag_improvement["metrics"]["RAG_Pipeline"]

    print("\n--- Task 3.2: Top-K Retrieval Sweep ---")

    N_SWEEP_SAMPLES = min(20, len(sql_eval_queries))

    if N_SWEEP_SAMPLES < len(sql_eval_queries):
        print(f"⚠️ Subsampling {N_SWEEP_SAMPLES} queries for top-k sweep to save time...")
        indices = random.sample(range(len(sql_eval_queries)), N_SWEEP_SAMPLES)

        sweep_queries = [sql_eval_queries[i] for i in indices]
        sweep_refs = [sql_eval_refs[i] for i in indices]
        sweep_baseline_preds = [sql_baseline_preds[i] for i in indices]
    else:
        sweep_queries = sql_eval_queries
        sweep_refs = sql_eval_refs
        sweep_baseline_preds = sql_baseline_preds

    k_values = [1, 2, 3, 5, 8]

    sweep_results = rag_evaluator.sweep_top_k(
        rag_pipeline=rag_pipeline,
        references=sweep_refs,
        queries=sweep_queries,
        baseline_preds=sweep_baseline_preds,
        k_values=k_values,
    )

    for k, res in sweep_results.items():
        print(f"  top_k={k}: metrics={res['metrics']} gain={res['gain_over_baseline']}")

    best_k = rag_evaluator.best_k(sweep_results, metric="BLEU")
    print(f"\n✅ Best top_k by BLEU gain (based on {N_SWEEP_SAMPLES} samples): {best_k}")

    rag_evaluator.plot_gain_vs_k(
        sweep_results,
        save_path=str(Path(config.outputs.plots_dir) / "topk_gain_sweep.png"),
    )

# ---------------------------------------------------------------
# Task (overall): Small Code LM vs. off-the-shelf LLM
# ---------------------------------------------------------------
print("\n--- Small Code LM vs. LLM Comparison ---")
try:
    llm_baseline = LargeLLMGenerator(config=config)
    llm_ref_name = llm_baseline.api_model if llm_baseline.backend == "api" else llm_baseline.local_model_name
    print(f"LLM backend: {llm_baseline.backend} ({llm_ref_name})")

    llm_prompt = f"User Question: {eval_query}\n\nAnswer: "
    llm_output = llm_baseline.generate(llm_prompt)
    print(f"LLM output:\n{llm_output[:500]}...")

    eval_device = config.evaluation.device if hasattr(config, "evaluation") else "cpu"
    llm_metrics = {
        "BLEU": round(EvaluationMetrics.compute_bleu([circle_area_ref], [llm_output]), 4),
        "CodeBERTScore": round(
            EvaluationMetrics.compute_bertscore([circle_area_ref], [llm_output], device=eval_device), 4
        ),
        "F1_Score": round(EvaluationMetrics.compute_f1([circle_area_ref], [llm_output]), 4),
        "Execution_Accuracy": round(
            EvaluationMetrics.evaluate_execution_accuracy([llm_output], test_assertions), 4
        ),
    }
    eval_results["LLM_Baseline"] = llm_metrics

    small_lm_metrics = eval_results.get("LoRA_Model", eval_results.get("Base_Model", {}))
    print("\nSmall Code LM (fine-tuned Qwen2.5-Coder-0.5B) vs. LLM baseline:")
    print(f"  Small_Code_LM -> {small_lm_metrics}")
    print(f"  LLM_Baseline  -> {llm_metrics}")
except Exception as e:
    print(f"⚠️ LLM baseline comparison skipped: {e}")

# Re-plot the full comparison now that RAG_Pipeline_SQL / LLM_Baseline are included.
visualizer.plot_comparison(eval_results, save_name="model_comparison_full.png")

print("\n✅ RAG improvement, top-K sweep, and small-code-LM-vs-LLM evaluation completed.")


--- Task 3.1: Measuring RAG Improvement (over the SQL eval set) ---


/Users/anjansuputra/miniforge3/envs/ApexEnv/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)



RAG vs. LoRA_Model (no-RAG) — gain per metric:
  BLEU                : -0.2651 ↓
  CodeBERTScore       : -0.0179 ↓
  F1_Score            : -0.3421 ↓
  ROUGE-1             : -0.3738 ↓
  ROUGE-L             : -0.3738 ↓

--- Task 3.2: Top-K Retrieval Sweep ---
  top_k=1: metrics={'BLEU': 0.0507, 'F1_Score': 0.1751, 'ROUGE-1': 0.1307, 'ROUGE-L': 0.1236} gain={'BLEU': -0.2833, 'F1_Score': -0.3647, 'ROUGE-1': -0.3856, 'ROUGE-L': -0.3927}
  top_k=2: metrics={'BLEU': 0.0773, 'F1_Score': 0.2167, 'ROUGE-1': 0.1695, 'ROUGE-L': 0.1695} gain={'BLEU': -0.2567, 'F1_Score': -0.3231, 'ROUGE-1': -0.3468, 'ROUGE-L': -0.3468}
  top_k=3: metrics={'BLEU': 0.0855, 'F1_Score': 0.2091, 'ROUGE-1': 0.1666, 'ROUGE-L': 0.1666} gain={'BLEU': -0.2485, 'F1_Score': -0.3307, 'ROUGE-1': -0.3497, 'ROUGE-L': -0.3497}
  top_k=5: metrics={'BLEU': 0.0842, 'F1_Score': 0.2417, 'ROUGE-1': 0.1813, 'ROUGE-L': 0.1813} gain={'BLEU': -0.2498, 'F1_Score': -0.2981, 'ROUGE-1': -0.335, 'ROUGE-L': -0.335}
  top_k=8: metrics={'BLEU': 0.0

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 7dd3a7a3-cc88-4839-afcb-cc938bdcd8e3)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-0.5B-Instruct/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


LLM output:
1. Import the math module to access the value of pi.
2. Define the function `circle_area` that takes one parameter, `radius`.
3. Inside the function, calculate the area using the formula `pi * radius^2`.
4. Return the calculated area.

Here's the complete code snippet:

```python
import math

def circle_area(radius):
    """
    Calculate the area of a circle given its radius.

    Parameters:
    radius (float): The radius of the circle.

    Returns:
    float: The area of the circle.
    """
...

Small Code LM (fine-tuned Qwen2.5-Coder-0.5B) vs. LLM baseline:
  Small_Code_LM -> {'BLEU': 0.0558, 'CodeBERTScore': 0.9288, 'F1_Score': 0.1852, 'ROUGE-1': 0.1351, 'ROUGE-L': 0.1351, 'Execution_Accuracy': 1.0, 'SQL_Exact_Match': 0.4, 'SQL_Execution_Accuracy': 0.6, 'SQL_BLEU': 0.334, 'SQL_CodeBERTScore': 0.9622}
  LLM_Baseline  -> {'BLEU': 0.0631, 'CodeBERTScore': 0.9458, 'F1_Score': 0.2083, 'Execution_Accuracy': 1.0}
✅ Score chart saved to: output/plots/scores_model_comparison_f